In [1]:
import os
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

In [2]:
RANDOM_SEED = 42
IMAGE_SIZE = (224, 224)
TOP_K = 5

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
DATASET_PATH = Path("/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset")
IMAGES_PATH = DATASET_PATH / "images"
STYLES_PATH = DATASET_PATH / "styles"
CSV_PATH = DATASET_PATH / "styles.csv"

In [4]:
# Loading Metadata

df = pd.read_csv(
    CSV_PATH,
    engine="python",
    on_bad_lines="skip"
)

print(f"Dataset Shape : {df.shape}")
df.head()

Dataset Shape : (44424, 10)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  object 
 2   masterCategory      44424 non-null  object 
 3   subCategory         44424 non-null  object 
 4   articleType         44424 non-null  object 
 5   baseColour          44409 non-null  object 
 6   season              44403 non-null  object 
 7   year                44423 non-null  float64
 8   usage               44107 non-null  object 
 9   productDisplayName  44417 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 3.4+ MB


In [6]:
# Missing Values

missing_values = df.isnull().sum()
missing_values

id                      0
gender                  0
masterCategory          0
subCategory             0
articleType             0
baseColour             15
season                 21
year                    1
usage                 317
productDisplayName      7
dtype: int64

In [7]:
print("Duplicate Rows :", df.duplicated().sum())

Duplicate Rows : 0


In [8]:
print("UNIQUE VALUES")
for column in df.columns:
    print(f"{column:<20}: {df[column].nunique()}")

UNIQUE VALUES
id                  : 44424
gender              : 5
masterCategory      : 7
subCategory         : 45
articleType         : 143
baseColour          : 46
season              : 4
year                : 13
usage               : 8
productDisplayName  : 31121


In [9]:
df.describe(include="all")

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
count,44424.000000,44424,44424,44424,44424,44409,44403,44423.000000,44107,44417
unique,NaN,5,7,45,143,46,4,NaN,8,31121
top,NaN,Men,Apparel,Topwear,Tshirts,Black,Summer,NaN,Casual,Lucera Women Silver Earrings
freq,NaN,22147,21397,15402,7067,9728,21472,NaN,34406,82
mean,29696.334301,NaN,NaN,NaN,NaN,NaN,NaN,2012.806497,NaN,NaN
std,17049.490518,NaN,NaN,NaN,NaN,NaN,NaN,2.126480,NaN,NaN
min,1163.000000,NaN,NaN,NaN,NaN,NaN,NaN,2007.000000,NaN,NaN
25%,14768.750000,NaN,NaN,NaN,NaN,NaN,NaN,2011.000000,NaN,NaN
50%,28618.500000,NaN,NaN,NaN,NaN,NaN,NaN,2012.000000,NaN,NaN
75%,44683.250000,NaN,NaN,NaN,NaN,NaN,NaN,2015.000000,NaN,NaN


In [10]:
image_files = list(IMAGES_PATH.glob("*.jpg"))
print(f"Total Images : {len(image_files)}")

Total Images : 44441


In [11]:
metadata_ids = set(df["id"].astype(str))
image_ids = {
    image.stem
    for image in image_files
}
print("Metadata IDs :", len(metadata_ids))
print("Image IDs    :", len(image_ids))

Metadata IDs : 44424
Image IDs    : 44441


In [12]:
missing_images = metadata_ids - image_ids
print("Missing Images :", len(missing_images))

Missing Images : 5


In [13]:
# Removing missing images

valid_ids = metadata_ids.intersection(image_ids)
df = df[df["id"].astype(str).isin(valid_ids)].copy()
print(df.shape)

(44419, 10)


In [14]:
# Fill missing values

df["baseColour"] = df["baseColour"].fillna("Unknown")
df["season"] = df["season"].fillna("Unknown")
df["usage"] = df["usage"].fillna("Unknown")
df["productDisplayName"] = df["productDisplayName"].fillna("Unknown")
df["year"] = df["year"].fillna(df["year"].median())

In [15]:
print(df.isnull().sum())

id                    0
gender                0
masterCategory        0
subCategory           0
articleType           0
baseColour            0
season                0
year                  0
usage                 0
productDisplayName    0
dtype: int64


In [16]:
df["year"] = df["year"].astype(int)

In [17]:
df["image_path"] = df["id"].astype(str).apply(
    lambda x: str(IMAGES_PATH / f"{x}.jpg")
)
df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans,/kaggle/input/datasets/paramaggarwal/fashion-p...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch,/kaggle/input/datasets/paramaggarwal/fashion-p...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants,/kaggle/input/datasets/paramaggarwal/fashion-p...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...


In [18]:
print(df["masterCategory"].value_counts())

masterCategory
Apparel           21392
Accessories       11274
Footwear           9219
Personal Care      2403
Free Items          105
Sporting Goods       25
Home                  1
Name: count, dtype: int64


In [19]:
print(df["subCategory"].value_counts().head(20))

subCategory
Topwear                     15398
Shoes                        7343
Bags                         3055
Bottomwear                   2693
Watches                      2542
Innerwear                    1808
Jewellery                    1079
Eyewear                      1073
Fragrance                    1011
Sandal                        963
Wallets                       933
Flip Flops                    913
Belts                         811
Socks                         698
Lips                          527
Dress                         478
Loungewear and Nightwear      470
Saree                         427
Nails                         329
Makeup                        307
Name: count, dtype: int64


In [20]:
print(df["articleType"].value_counts().head(20))

articleType
Tshirts                  7066
Shirts                   3215
Casual Shoes             2845
Watches                  2542
Sports Shoes             2036
Kurtas                   1844
Tops                     1762
Handbags                 1759
Heels                    1323
Sunglasses               1073
Wallets                   936
Flip Flops                914
Sandals                   897
Briefs                    849
Belts                     813
Backpacks                 724
Socks                     686
Formal Shoes              637
Perfume and Body Mist     613
Jeans                     608
Name: count, dtype: int64


In [21]:
print(df["gender"].value_counts())

gender
Men       22142
Women     18631
Unisex     2161
Boys        830
Girls       655
Name: count, dtype: int64


In [22]:
print(df["season"].value_counts())

season
Summer     21470
Fall       11431
Winter      8515
Spring      2982
Unknown       21
Name: count, dtype: int64


In [23]:
print(df["baseColour"].value_counts().head(20))

baseColour
Black        9727
White        5538
Blue         4917
Brown        3494
Grey         2741
Red          2453
Green        2115
Pink         1860
Navy Blue    1789
Purple       1640
Silver       1090
Yellow        778
Beige         749
Gold          628
Maroon        581
Orange        530
Olive         410
Multi         394
Cream         389
Steel         315
Name: count, dtype: int64


In [24]:
# Saving clean metadata

OUTPUT_PATH = Path("/kaggle/working")
clean_csv = OUTPUT_PATH / "clean_metadata.csv"
df.to_csv(clean_csv, index=False)
print("Saved Successfully")
print(clean_csv)

Saved Successfully
/kaggle/working/clean_metadata.csv


In [25]:
print(f"Final Dataset Shape : {df.shape}")
print(f"Total Images        : {len(image_files)}")
print(f"Missing Images      : {len(missing_images)}")
print(f"Duplicate Rows      : {df.duplicated().sum()}")

Final Dataset Shape : (44419, 11)
Total Images        : 44441
Missing Images      : 5
Duplicate Rows      : 0
